# GT+ML+NLP ideas

**Багатовимірний аналіз векторних представлень тексту із застосуванням машинного навчання та змагальних моделей**

1. Unrolled GAN
2. Prediction methods (зокрема Optimistic Mirror Descent)
3. Wasserstein GAN (WGAN-GP)
4. Prediction methods, Lookahead-minmax


# "Мінмаксні принципи та сідлові точки у просторах семантичних представлень тексту"


## 1. Adversarial Word Embeddings як гра двох гравців

Класичний Word2Vec шукає

$$
\max_{\theta} L(\theta)
$$

де $\theta$ — embedding-и слів.

Але можна побудувати гру: генератор створює embedding-и; дискримінатор намагається визначити, чи вони "природні"

Тоді отримуємо

$$
\min_D \max_G V(G,D)
$$

як у GAN.

Класичний Word2Vec навчається так:

Для центрального слова (w) потрібно зробити embedding

$$
e_w\in\mathbb R^d
$$

таким, щоб він добре передбачав контекст.

Наприклад, Skip-Gram максимізує

$$
L(E)=
\sum_{(w,c)}
\log P(c|w).
$$

Тобто ми просто шукаємо

$$
\max_E L(E).
$$

Нехай тепер існує

* **Generator** (G)
* **Adversary** (A)

Generator будує embedding

$$ e=G(w). $$

Після цього adversary намагається його "зіпсувати".

Наприклад

$$ \tilde e=e+\delta. $$

де $\delta$ — спеціально знайдене збурення.

Generator хоче, щоб навіть після такого збурення embedding залишався семантично правильним.

Adversary хоче, максимально його зруйнувати.

Отримуємо гру

$$
\min_A
\max_G
V(G,A).
$$

**Що означає "зіпсувати embedding"?**

Наприклад,

Generator видає

```
cat
```

↓

```
[0.42
-0.11
...
]
```

Adversary додає

```
+0.03
-0.07
...
```

Після цього найближчим словом стає

```
dog
```

або навіть

```
car
```

Generator програв.


**Цикл навчання**

```
Generator

↓

Embedding

↓

Adversary

↓

Perturbed embedding

↓

Context prediction

↓

Generator update

↓

Adversary update
```

**Варіанти**: семантична відстань (множина найближчих сусідів embedding, штрафуємо за зміну сусідів), Cosine Similarity (мінімізуємо $\cos(e,\tilde e)$), Класифікація (Embedding подаємо у класифікатор, generator хлче залишити positive, adv. - negative).

**Як реалізувати Generator**: 
- Generator звичайний Word2Vec, Він виробляє embedding.
- Generator — невеликий MLP

**Adversary**: не генерує слова, генерує $\delta$, Embedding -> NN -> $\delta$ -> embedding + $\delta$.




## 2. Embedding Space як многовид

Сучасні embedding-и (BERT, SBERT, E5, GTE) живуть у просторах

$$
\mathbb R^{384},
\mathbb R^{768},
\mathbb R^{1024}
$$

Але реально вони лежать на деякому многовиді меншої розмірності.

Можна розглядати задачу:

$$ \min_x \max_y f(x,y) $$

де (x) — напрямок семантичної зміни, (y) — напрямок шуму, та досліджувати сідлові точки.

## 3. Теорія ігор для семантичної стабільності



## 4. Аналог теореми фон Неймана для embedding-просторів



## 5. GAN для побудови кращих embedding-ів



## 6. Adversarial Attacks на Embeddings


## 7. Wasserstein GAN + текстові представлення



## 8. Fuzzy-GAN для текстових представлень



In [1]:
# Adversarial Word Embeddings as the game of two players

import numpy as np
from scipy.optimize import differential_evolution
from sentence_transformers import SentenceTransformer
from numpy.linalg import norm

model = SentenceTransformer("all-MiniLM-L6-v2")

sentence = "The patient suffers from fever and cough."

embedding = model.encode(sentence)
embedding = embedding.astype(np.float64)

def cosine(a,b):
    return np.dot(a,b)/(norm(a)*norm(b))

EPS = 0.5 # limitation

# loss
def objective(delta):
    delta=np.array(delta)
    if norm(delta)>EPS:
        return 1000
    perturbed=embedding+delta
    similarity=cosine(
        embedding,
        perturbed
    )

    return similarity

# optimization
bounds=[(-0.2,0.2)]*embedding.shape[0]

result=differential_evolution(
    objective,
    bounds,
    maxiter=40,
    popsize=10,
    seed=42
)

delta=result.x

perturbed=embedding+delta

print(cosine(embedding, perturbed))


# find closest sentences
corpus=[
"The patient has influenza.",
"The patient suffers from pneumonia.",
"The weather is sunny today.",
"Artificial intelligence is transforming medicine.",
"The patient reports cough and fever.",
"He likes football."
]

corpus_embeddings=model.encode(corpus)

# search function
def nearest(vector,k=3):
    similarities=[]
    for i,e in enumerate(corpus_embeddings):
        sim=cosine(vector,e)
        similarities.append((sim,corpus[i]))
    similarities.sort(reverse=True)

    return similarities[:k]

print("Original")

for sim,text in nearest(embedding):
    print(sim,text)

print()

print("Perturbed")

for sim,text in nearest(perturbed):
    print(sim,text)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\personal\sci\nlp-fuzzy\src\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vmelnyk2\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

0.42007926351365577
Original
0.9402774739355119 The patient reports cough and fever.
0.7212866195046193 The patient has influenza.
0.6998644295605936 The patient suffers from pneumonia.

Perturbed
0.38940227980578856 The patient reports cough and fever.
0.3115835266792639 The patient has influenza.
0.2217802748870587 The patient suffers from pneumonia.


In [3]:
import torch
import torch.nn as nn


class Generator(nn.Module):

    def __init__(self, dim):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(dim, 256),
            nn.ReLU(),

            nn.Linear(256,256),
            nn.ReLU(),

            nn.Linear(256,dim)

        )

    def forward(self,x):

        return self.net(x)
    
generator = Generator(embedding.shape[0])

optimizer = torch.optim.Adam(
    generator.parameters(),
    lr=1e-3
)

def attack(current_embedding):

    def objective(delta):

        delta=np.array(delta)

        if norm(delta)>EPS:
            return 1000

        perturbed=current_embedding+delta

        recovered=generator(
            torch.tensor(
                perturbed,
                dtype=torch.float32
            )
        ).detach().numpy()

        return cosine(
            embedding,
            recovered
        )

    bounds=[(-0.2,0.2)]*len(current_embedding)

    result=differential_evolution(
        objective,
        bounds,
        maxiter=20,
        popsize=10
    )

    return result.x

loss_fn = nn.CosineEmbeddingLoss()


def train_generator(perturbed):

    optimizer.zero_grad()

    x=torch.tensor(
        perturbed,
        dtype=torch.float32
    )

    target=torch.tensor(
        embedding,
        dtype=torch.float32
    )

    recovered=generator(x)

    label=torch.tensor([1.0])

    loss=loss_fn(

        recovered.unsqueeze(0),

        target.unsqueeze(0),

        label

    )

    loss.backward()

    optimizer.step()

    return loss.item()

# THE GAME LOOP
for iteration in range(30):

    #########################
    # Player 2 attacks
    #########################

    delta=attack(embedding)

    perturbed=embedding+delta

    #########################
    # Player 1 responds
    #########################

    loss=train_generator(perturbed)

    recovered=generator(
        torch.tensor(
            perturbed,
            dtype=torch.float32
        )
    ).detach().numpy()

    print()

    print("Iteration",iteration)

    print("Attack cosine",
          cosine(embedding,perturbed))

    print("Recovered cosine",
          cosine(embedding,recovered))

    print("Loss",loss)




Iteration 0
Attack cosine 0.4111501124648654
Recovered cosine 0.25224861025423345
Loss 0.9346866011619568

Iteration 1
Attack cosine 0.3724551196597018
Recovered cosine 0.3865772105509629
Loss 0.7870599031448364

Iteration 2
Attack cosine 0.3663445476559098
Recovered cosine 0.4969498288660987
Loss 0.6549496650695801

Iteration 3
Attack cosine 0.4351830388469834
Recovered cosine 0.5698141437073375
Loss 0.5654144287109375

Iteration 4
Attack cosine 0.3972666570657492
Recovered cosine 0.6717743834103528
Loss 0.44443780183792114

Iteration 5
Attack cosine 0.4485120311960027
Recovered cosine 0.739936831863092
Loss 0.35888808965682983

Iteration 6
Attack cosine 0.37385707819820774
Recovered cosine 0.7922742571482159
Loss 0.2867857813835144

Iteration 7
Attack cosine 0.38843956750830017
Recovered cosine 0.8446834396632787
Loss 0.22077006101608276

Iteration 8
Attack cosine 0.46151238957647633
Recovered cosine 0.8886673554096909
Loss 0.1535741686820984

Iteration 9
Attack cosine 0.40570407638